# Calculate trades

This notebook shows one practical workflow for building a multi-year MARIO database from repeated parser calls and then using `db.calc_trades(...)` to compare region-by-region trade patterns across scenarios.

The example uses EXIOBASE 3.10.1 monetary IOT files stored locally as yearly zip archives. It is intentionally kept as a workflow example rather than one notebook executed during the documentation build.

In [ ]:
import mario

## Define the local EXIOBASE time-series inputs

The new parser-scenario workflow lets one MARIO database instance hold multiple years as separate scenarios. The first parsed year initializes the database, then the remaining years are imported as new scenarios.

In [ ]:
FOLDER = '/Users/lorenzorinaldi/Library/CloudStorage/OneDrive-SharedLibraries-PolitecnicodiMilano/DENG-SESAM - Documenti/c-Research/a-Datasets/_Input Output Databases/EXIOBASE/3.10.1'
TABLE = 'IOT'
YEARS = range(2012, 2023)
SYSTEM = 'pxp'

# Uncomment and run if you need to download the database.
# You can also download it manually from https://zenodo.org/records/20051562
# mario.download_exiobase(version='3.10.1', system=SYSTEM, years=YEARS, path=FOLDER)

## Initialize the first year and rename the baseline scenario

The first parse creates the database in the standard way. Then `baseline` is renamed to the actual year so that all yearly scenarios follow one consistent naming convention.

In [ ]:
db = mario.parse_exiobase(
    path=f'{FOLDER}/{TABLE}_{YEARS[0]}_{SYSTEM}.zip',
    table='IOT',
    unit='Monetary',
)
db.rename_scenario('baseline', YEARS[0])

## Import the remaining years as scenarios

Each additional parser call reuses the same `Database` instance and imports one new compatible scenario instead of creating a separate database object. Only the parser raw blocks are imported; derived matrices can still be computed scenario by scenario when needed.

In [ ]:
for year in YEARS[1:]:
    print(f'Processing year {year}...')
    db.parse_exiobase(
        path=f'{FOLDER}/{TABLE}_{year}_{SYSTEM}.zip',
        table='IOT',
        unit='Monetary',
        new_scenario=year,
    )
    print('DONE')

## Calculate and export yearly trade heatmaps for one sector

`db.calc_trades(...)` aggregates one sector or commodity into an origin-by-destination trade matrix, and can also save the corresponding heatmap directly to HTML or image files.

![How calc_trades works](../../_static/images/calc_trades.png)

The next loop calculates one region-by-region trade matrix for `Nickel ores and concentrates` in each scenario and writes the corresponding heatmap to one HTML file per year.

In [ ]:
sector = 'Nickel ores and concentrates'

trades = {}
for year in YEARS:
    trades[year] = db.calc_trades(
        item=sector,
        total=True,
        scenario=str(year),
        save_plot=f'plots/{year}_EXIOBASE.html',
        title=f'{sector} - {year} - From EXIOBASE 3.10.1',
    )

## Visual comparison: 2013 vs 2014

The two static heatmaps below were exported from the same workflow for 2013 and 2014. They illustrate one substantive supply-chain shift: the Indonesia-to-China flow of nickel ores becomes much less prominent after 2013, consistent with Indonesia's policy move to stop exporting raw nickel ores.

### 2013

![Nickel ores trade heatmap for 2013](../../_static/images/supply_chain_trades_2013.png)

### 2014

![Nickel ores trade heatmap for 2014](../../_static/images/supply_chain_trades_2014.png)

In this pair of plots, the `ID -> CN` cell is much more visible in 2013 than in 2014. Even in this simple visual form, `db.calc_trades(...)` is already useful for tracking discontinuities in supply-chain geography before moving to more structured indicators.